<!-- launch-badges -->
<a href="https://colab.research.google.com/github/laban254/ml-for-infrastructure/blob/main/01_foundations/distributed_data/pyspark_log_processing.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
&nbsp;
<a href="https://mybinder.org/v2/gh/laban254/ml-for-infrastructure/main?urlpath=lab/tree/01_foundations/distributed_data/pyspark_log_processing.ipynb" target="_blank"><img src="https://mybinder.org/badge_logo.svg" alt="Open in Binder"/></a>

> ▶️ **Run this notebook live** — no install needed. Click a badge above to open it in a free cloud runtime.

# Distributed Data Processing with PySpark

## Objectives
- Bridge the gap between local tools (Pandas) and infrastructure-level big data processing (`PySpark`).
- Understand the map-reduce paradigm for processing logs or metrics that won't fit into your local machine's RAM.
- Set up a local Spark Session to aggregate and query dataset sizes that typically require a cluster.

## Expected Outcome
- A functional local PySpark pipeline capable of grouping and extracting metrics from millions of log rows.

## Challenge
- Rewrite a standard Pandas DataFrame `groupby()` utilizing PySpark RDDs or Spark DataFrames.

In [1]:
# !pip install pyspark

### 1. Initializing Spark
Unlike Pandas, Spark requires an active "Session" or "Context" that connects your Python code to the JVM (Java Virtual Machine) backend that does the heavy lifting.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, month, count, avg

# Set up a local spark session using all available CPU cores (*)
spark = SparkSession.builder \
    .appName("InfraLogAnalysis") \
    .master("local[*]") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/17 15:56:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### 2. Simulating a Gigantic Log File
We'll generate a dummy dataset of server events that resembles a real-world ELK stack export.

In [3]:
import pandas as pd
import numpy as np

# Create 500,000 synthetic log records
num_records = 500000
sample_services = ['auth-service', 'billing-api', 'frontend-ui', 'database-pg']
sample_levels = ['INFO', 'WARN', 'ERROR', 'DEBUG']

data = {
    "timestamp": pd.date_range(start="2025-01-01", periods=num_records, freq="15S"),
    "service": np.random.choice(sample_services, num_records, p=[0.4, 0.2, 0.3, 0.1]),
    "log_level": np.random.choice(sample_levels, num_records, p=[0.7, 0.1, 0.05, 0.15]),
    "response_time_ms": np.random.gamma(shape=2.0, scale=50.0, size=num_records)
}

# Convert Pandas DataFrame to PySpark DataFrame
pdf = pd.DataFrame(data)
df = spark.createDataFrame(pdf)

print("Spark DataFrame Schema:")
df.printSchema()

/tmp/ipykernel_399429/135687121.py:10: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  "timestamp": pd.date_range(start="2025-01-01", periods=num_records, freq="15S"),


Spark DataFrame Schema:
root
 |-- timestamp: timestamp (nullable = true)
 |-- service: string (nullable = true)
 |-- log_level: string (nullable = true)
 |-- response_time_ms: double (nullable = true)



### 3. Distributed Queries (Lazy Evaluation)
In Spark, defining a query doesn't execute it immediately. Execution happens only when an action (like `.show()` or `.collect()`) is called.

In [4]:
# Aggregating average response times and error counts by service
# If this was a 100GB dataset, this query would farm out to your cluster implicitly
service_metrics = df.filter(col("log_level").isin(["ERROR", "WARN"])) \
    .groupBy("service", "log_level") \
    .agg(
        count("*").alias("total_occurrences"),
        avg("response_time_ms").alias("avg_latency_ms")
    ) \
    .orderBy("total_occurrences", ascending=False)

print("Top Services with Warnings and Errors:")
service_metrics.show()

Top Services with Warnings and Errors:


26/06/17 15:57:23 WARN TaskSetManager: Stage 0 contains a task of very large size (2609 KiB). The maximum recommended task size is 1000 KiB.


+------------+---------+-----------------+------------------+
|     service|log_level|total_occurrences|    avg_latency_ms|
+------------+---------+-----------------+------------------+
|auth-service|     WARN|            20140|100.20615912591839|
| frontend-ui|     WARN|            14829| 99.88536340446106|
|auth-service|    ERROR|            10074| 99.11996909678966|
| billing-api|     WARN|             9850| 99.65755027981766|
| frontend-ui|    ERROR|             7450|  99.2483431298239|
| database-pg|     WARN|             5035|101.56992637097139|
| billing-api|    ERROR|             4922|101.31928906234492|
| database-pg|    ERROR|             2483| 100.4100630008913|
+------------+---------+-----------------+------------------+



In [5]:
# Stop the Spark context to free up memory
spark.stop()